Connect to GPU environment. How to do it:
1. Go to the 'Runtime' menu at the top.
2. Select 'Change runtime type'.
3. Choose Runtime type 'Python 3' and Hardware accelerator 'T4 GPU' (or another GPU) from the dropdown.
4. Click 'Save'.
5. At the top right click "Connect" or just run the first cell or all cells.

In [ ]:
# Verify that GPU environment is activated

import torch

if torch.cuda.is_available():
    print(f"✅ Success! Connected to GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU not detected.")
    print("-" * 30)
    print("INSTRUCTIONS TO FIX:")
    print("1. Go to the 'Runtime' menu at the top.")
    print("2. Select 'Change runtime type'.")
    print("3. Choose Runtime type 'Python 3' and Hardware accelerator 'T4 GPU' (or another GPU) from the dropdown.")
    print("4. Click 'Save' and re-run this cell.")
    print("-" * 30)

In [ ]:
# Repo Path Setup

from pathlib import Path

platform_type = 'colab_linux'

# Set repo variables
github_username = 'artem-ryzhov-1'
repo_name = 'dqd-lzsm-simulator'
branch_name = 'main'

repo_path = Path('/content') / repo_name

repo_path.mkdir(parents=True, exist_ok=True)

parent_dir = repo_path.parent

In [ ]:
# Clone Repository from GitHub

import subprocess

if not any(repo_path.iterdir()):
    print("Cloning repository...")
    clone_url = f"https://github.com/{github_username}/{repo_name}.git"
    
    git_command = [
        'git', 'clone', '--branch', branch_name, '--single-branch',
        clone_url,
        str(repo_path)
    ]
    try:
        result = subprocess.run(
            git_command,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("STDOUT:", e.stdout)
        print("STDERR:", e.stderr)
        raise
    
    # Verify current branch
    result = subprocess.run(
        ['git', 'rev-parse', '--abbrev-ref', 'HEAD'],
        cwd=repo_path,
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    print(f"Current branch: {result.stdout.strip()}")
    print("Repository cloned successfully.")
else:
    print("Repository already has files or directories. Skipping clone.")

# Change to parent directory
from IPython import get_ipython
get_ipython().run_line_magic('cd', str(parent_dir))

In [ ]:
# Add the repo root folder to sys.path if it is not already included

import sys

if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path)) # Ensures 'python' folder is accessible for imports
    print(f"✅ Sys path updated: {repo_path}")

In [ ]:
# Compile the CUDA code

from app.python import helpers

helpers.run_build_script(repo_path, platform_type)

In [ ]:
# Install Python dependencies

helpers.install_dependencies(repo_path, platform_type)

In [ ]:
# Verify that all required files are present

helpers.verify_required_files(repo_path)

In [ ]:
# Launch the interactive dashboard

# Enable Panel and HoloViews extensions
import panel as pn
pn.extension('mathjax')
import holoviews as hv
hv.extension('bokeh')
import os, sys, importlib

# Change to python directory
os.chdir(repo_path / "app" / "python")

importlib.reload(sys.modules['app'])
from app.python import helpers
from app.python.app_class_interactive_interferogram_dynamics import InteractiveInterferogramDynamics

# Check whether cupy and cudf libraries are available
CUPY_CUDF_AVAILABLE = helpers.detect_cupy_cudf()

# Chose one of the render mode options. Options: 'raster_dynamic', 'vector', 'raster_static', 'raster_static_gpu', 'raster_dynamic', 'raster_dynamic_gpu'
render_mode = helpers.modify_render_mode('raster_dynamic', CUPY_CUDF_AVAILABLE)

# Create app instance
app = InteractiveInterferogramDynamics(
    eps0_min=-0.006,
    eps0_max=0.006,
    A_min=0.0,
    A_max=0.01,
    N_points_target=500_000,
    delta_C_range=(0, 0.001),
    GammaL0_range=(0, 1000),
    GammaR0_range=(0, 150),
    Gamma_eg0_range=(0, 50),
    Gamma_phi0_range=(0, 100),
    sigma_eps_range=(0, 0.001),
    nu_range=(0, 40),
    E_C_range=(0.01, 0.4),
    N_steps_period_array=(100, 2000),
    N_periods_array=(1, 20),
    N_periods_avg_array=(1, 10),
    N_samples_noise_array=(0, 1000),
    delta_C_default=0.0003,
    GammaL0_default=560.0,
    GammaR0_default=80.0,
    Gamma_eg0_default=30.0,
    Gamma_phi0_default=55.0,
    sigma_eps_default=0.00015,
    nu_default=21.0,
    E_C_default=0.195,
    N_steps_period_default=1000,
    N_periods_default=10,
    N_periods_avg_default=1,
    N_samples_noise_default=10,
    dC_default_thresholds=(-0.01, 0.005),
    rho_init = [0.25, 0.25, 0.25, 0.25],

    platform_type=platform_type,
    repo_path=repo_path,
    cmap_name='fire',
    render_mode=render_mode
)

# Create dashboard
dashboard = app.create_dashboard()

# Launch dashboard
helpers.launch_app_colab(dashboard)